[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/06_export_onnx.ipynb)

# Notebook 6 — ONNX Export

Merge the LoRA adapter into the base model, export to ONNX, and publish to `spatialft/LFM2-350M-StepGame-ONNX` on HuggingFace Hub.

That repo can be loaded directly in `lfm2-web` by setting `MODEL_ID = 'spatialft/LFM2-350M-StepGame-ONNX'`.

**Prerequisites:**
- Notebook 03 must have run and published the LoRA adapter to `results/finetuned/lora_adapter/`
- Colab secrets: `HF_TOKEN` (write access to the `spatialft` HF org)

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/spatialft.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/spatialft/spatialft.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.colab_utils import prepare_notebook

REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
print(f'Repo ready at {REPO}')

In [ ]:
# Install ONNX export deps (separate from main requirements — not needed for training/eval)
# onnxruntime>=1.18 required for Python 3.12 and MatMulNBitsQuantizer (renamed from
# MatMul4BitsQuantizer in 1.17). Install here so the kernel restart picks up the new version.
import subprocess, sys, os
sentinel = Path('/tmp/spatialft_notebook06_onnx_deps')
if not sentinel.exists():
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'optimum[onnxruntime]>=1.19', 'onnx>=1.16', 'onnxconverter-common>=1.14', 'onnxscript',
        'onnxruntime>=1.18',
    ], check=True)
    sentinel.write_text('ok')
    print('Dependencies installed. Restarting runtime...')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies ready.')

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_ID = 'LiquidAI/LFM2-350M'
HF_REPO_ID    = None  # set below after reading HF_USERNAME from token

In [ ]:
# Verify adapter exists (requires notebook 03 to have run)
adapter_dir = PATHS['adapter_dir']
if not adapter_dir.exists():
    raise FileNotFoundError(
        f'LoRA adapter not found at {adapter_dir}.\n'
        'Run notebook 03 and publish the adapter first.'
    )
print(f'Adapter found: {adapter_dir}')

In [ ]:
# Load base model + merge adapter
# Tokenizer loaded from base model — the adapter copy references TokenizersBackend
# which isn't available in all environments.
print('Loading base model...')
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float32,  # fp32 for clean ONNX export; quantize after
    device_map='cpu',
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print('Merging LoRA adapter...')
model = PeftModel.from_pretrained(base, str(adapter_dir))
model = model.merge_and_unload()
model.eval()
print('Merge complete.')

In [ ]:
# Save merged weights locally (needed by optimum for ONNX export)
MERGED_DIR = PATHS['results_root'] / 'finetuned' / 'export' / 'merged_hf'
MERGED_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f'Merged model saved to {MERGED_DIR}')

In [ ]:
import onnx, os
import numpy as np
from huggingface_hub import HfApi, hf_hub_download
from onnx import numpy_helper

ONNX_DIR = PATHS['results_root'] / 'finetuned' / 'export' / 'onnx'
ONNX_DIR.mkdir(parents=True, exist_ok=True)
BASE_ONNX_REPO = 'onnx-community/LFM2-350M-ONNX'

# 1. Find files
api = HfApi()
files = list(api.list_repo_files(BASE_ONNX_REPO))
onnx_files = [f for f in files if f.endswith('.onnx')]
fp32_file = next(f for f in onnx_files if 'q4' not in f and 'fp16' not in f)
onnx_folder = str(Path(fp32_file).parent)

# 2. Download fp32 model + external data
sibling_files = [f for f in files if f.startswith(onnx_folder + '/') and 'q4' not in f and 'fp16' not in f]
print(f'Downloading {len(sibling_files)} files...')
for f in sibling_files:
    hf_hub_download(BASE_ONNX_REPO, f)

# 3. Load with external data
base_onnx_path = hf_hub_download(BASE_ONNX_REPO, fp32_file)
base_dir = os.path.dirname(base_onnx_path)
model_onnx = onnx.load(base_onnx_path, load_external_data=False)
onnx.load_external_data_for_model(model_onnx, base_dir)
print(f'Loaded — {len(model_onnx.graph.initializer)} initializers')

# 4. Build finetuned state dict
finetuned_state = {k: v.cpu().float().numpy() for k, v in model.state_dict().items()}

# 5. Inspect name mapping
onnx_names = {init.name for init in model_onnx.graph.initializer}
pt_names = set(finetuned_state.keys())
matched = onnx_names & pt_names
print(f'Name overlap: {len(matched)} / {len(onnx_names)} ONNX initializers match PyTorch keys directly')
print('\nSample ONNX names:')
for init in model_onnx.graph.initializer[:3]:
    print(f'  {init.name}')
print('\nSample PyTorch keys:')
for k in list(finetuned_state.keys())[:3]:
    print(f'  {k}')

# 6. Replace weights
replaced, skipped = 0, []
for init in model_onnx.graph.initializer:
    if init.name in finetuned_state:
        ft = finetuned_state[init.name]
        base = numpy_helper.to_array(init)
        if ft.shape == base.shape:
            init.CopyFrom(numpy_helper.from_array(ft.astype(base.dtype), name=init.name))
            replaced += 1
        else:
            skipped.append(f'{init.name}: shape mismatch {base.shape} vs {ft.shape}')
    else:
        skipped.append(f'{init.name}: not found in finetuned state')

print(f'\nReplaced: {replaced} / {len(model_onnx.graph.initializer)}')
if skipped:
    print(f'Skipped ({len(skipped)}):')
    for s in skipped[:5]: print(f'  {s}')

# 7. Save (inline weights — no external data file needed)
out_path = ONNX_DIR / 'model.onnx'
onnx.save(model_onnx, str(out_path))
print(f'\nmodel.onnx written — {out_path.stat().st_size / 1e6:.1f} MB')

In [ ]:
# fp16 conversion
import onnx
from onnxconverter_common import float16

print('Converting to fp16...')
m = onnx.load(str(ONNX_DIR / 'model.onnx'))
onnx.save(float16.convert_float_to_float16(m), str(ONNX_DIR / 'model_fp16.onnx'))
print('model_fp16.onnx written')

In [ ]:
# Q4 weight injection — preserves exact MatMulNBits op format from base q4 model.
#
# ORT-quantizer approaches fail: ORT 1.24 produces MatMulNBits attributes that
# Transformers.js's bundled ORT build doesn't recognise. Injecting directly into
# the base q4 ONNX guarantees compatibility — same op, same attribute schema.
#
# Strategy:
#   1. Load base q4 ONNX (known-working with Transformers.js)
#   2. Map each MatMulNBits node's weight slot → PyTorch key via graph position
#      (fp32 base and q4 base share the same graph topology; MatMul[i] ↔ MatMulNBits[i])
#   3. Re-quantize finetuned fp32 weight → uint4 packed format, inject
#   4. Direct-swap remaining initializers (embeddings, layer norms)

import os
import onnx
import numpy as np
from onnx import numpy_helper
from huggingface_hub import HfApi, hf_hub_download

BASE_ONNX_REPO = 'onnx-community/LFM2-350M-ONNX'

# --- Download base q4 ONNX + its external data sidecar ---
api_files = list(HfApi().list_repo_files(BASE_ONNX_REPO))
q4_src_file = next(f for f in api_files if f.endswith('.onnx') and 'q4' in f)
q4_folder   = str(Path(q4_src_file).parent)

# Download all files in the same folder that are related to the q4 model
q4_siblings = [f for f in api_files
               if f.startswith(q4_folder + '/') and 'q4' in f]
print(f'Downloading {len(q4_siblings)} q4 files: {q4_siblings}')
for f in q4_siblings:
    hf_hub_download(BASE_ONNX_REPO, f)

q4_src_path = hf_hub_download(BASE_ONNX_REPO, q4_src_file)
q4_base_dir = os.path.dirname(q4_src_path)

print(f'Loading base q4: {q4_src_file}')
model_q4 = onnx.load(q4_src_path, load_external_data=False)
onnx.load_external_data_for_model(model_q4, q4_base_dir)
print(f'Loaded — {len(model_q4.graph.initializer)} initializers')

# --- Build index structures ---
q4_init = {init.name: init for init in model_q4.graph.initializer}

# Ordered MatMul weight names from fp32 model (same topology as q4)
def matmul_weight_names(m):
    init_set = {i.name for i in m.graph.initializer}
    return [n.input[1] for n in m.graph.node
            if n.op_type == 'MatMul' and len(n.input) >= 2 and n.input[1] in init_set]

# Ordered (weight_init, scale_init) from q4 MatMulNBits nodes
def matmulnbits_slots(m):
    init_set = {i.name for i in m.graph.initializer}
    return [(n.input[1], n.input[2]) for n in m.graph.node
            if n.op_type == 'MatMulNBits' and len(n.input) >= 3 and n.input[1] in init_set]

def get_block_size(m):
    for n in m.graph.node:
        if n.op_type == 'MatMulNBits':
            for a in n.attribute:
                if a.name == 'block_size':
                    return a.i
    return 32

fp32_weights = matmul_weight_names(model_onnx)
q4_slots     = matmulnbits_slots(model_q4)
block_size   = get_block_size(model_q4)

print(f'fp32 MatMul weights:    {len(fp32_weights)}')
print(f'q4  MatMulNBits slots:  {len(q4_slots)}')
print(f'block_size:             {block_size}')
assert len(fp32_weights) == len(q4_slots), 'topology mismatch between fp32 and q4 base models'

# --- Quantization helper ---
def quantize_fp32_to_uint4(weight_fp32, block_size):
    """
    Quantize [K, N] fp32 weight to MatMulNBits uint4 packed format.
    Returns (packed_uint8 [N, K_padded//2], scales [N*n_blocks])
    """
    K, N = weight_fp32.shape
    n_blocks_k = (K + block_size - 1) // block_size
    K_padded   = n_blocks_k * block_size

    w = weight_fp32.T.copy()                              # [N, K]
    if K_padded > K:
        w = np.pad(w, ((0, 0), (0, K_padded - K)))

    w_blocks = w.reshape(N, n_blocks_k, block_size)

    # Symmetric int4: range [-8, 7], stored offset as [0, 15]
    abs_max = np.max(np.abs(w_blocks), axis=-1)           # [N, n_blocks_k]
    scales  = (abs_max / 7.0).astype(np.float32)
    s_safe  = np.where(scales > 0, scales, 1.0)[:, :, np.newaxis]
    w_q     = np.clip(np.round(w_blocks / s_safe), -8, 7).astype(np.int32) + 8
    w_q     = w_q.astype(np.uint8)

    # Pack pairs of nibbles: low nibble = even index, high nibble = odd index
    w_packed = (w_q[:, :, ::2] & 0xF) | ((w_q[:, :, 1::2] & 0xF) << 4)
    return w_packed.reshape(N, -1), scales.reshape(-1)

# --- Inject MatMulNBits weights ---
replaced_q4, skipped = 0, []

for pt_key, (q4_w_name, q4_s_name) in zip(fp32_weights, q4_slots):
    if pt_key not in finetuned_state:
        skipped.append(f'{pt_key}: not in finetuned state')
        continue
    ft     = finetuned_state[pt_key]                      # [K, N] fp32
    base_w = numpy_helper.to_array(q4_init[q4_w_name])
    base_s = numpy_helper.to_array(q4_init[q4_s_name])

    packed, scales = quantize_fp32_to_uint4(ft, block_size)
    scales = scales.astype(base_s.dtype)                  # match base dtype

    if packed.shape != base_w.shape:
        skipped.append(f'{pt_key}: packed shape {packed.shape} != base {base_w.shape}')
        continue

    q4_init[q4_w_name].CopyFrom(numpy_helper.from_array(packed, name=q4_w_name))
    q4_init[q4_s_name].CopyFrom(numpy_helper.from_array(scales, name=q4_s_name))
    replaced_q4 += 1

print(f'\nMatMulNBits replaced: {replaced_q4} / {len(fp32_weights)}')

# --- Direct-swap remaining initializers (embeddings, layer norms, etc.) ---
matmul_weight_set = set(fp32_weights)
replaced_direct   = 0

for init in model_q4.graph.initializer:
    if init.name in matmul_weight_set:
        continue                                          # already handled
    if init.name in finetuned_state:
        ft   = finetuned_state[init.name]
        base = numpy_helper.to_array(init)
        if ft.shape == base.shape:
            init.CopyFrom(numpy_helper.from_array(ft.astype(base.dtype), name=init.name))
            replaced_direct += 1
        else:
            skipped.append(f'{init.name}: shape mismatch {ft.shape} vs {base.shape}')

print(f'Direct replaced:      {replaced_direct}')
if skipped:
    print(f'Skipped ({len(skipped)}):')
    for s in skipped[:10]: print(f'  {s}')

# Save with inlined weights (no external data sidecar needed for upload)
q4_out = ONNX_DIR / 'model_q4.onnx'
onnx.save(model_q4, str(q4_out))
print(f'\nmodel_q4.onnx written — {q4_out.stat().st_size / 1e6:.1f} MB')

In [ ]:
import shutil
from google.colab import userdata
from huggingface_hub import HfApi, login, upload_file

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Missing Colab secret HF_TOKEN.')

login(token=HF_TOKEN)
api = HfApi()

HF_REPO_ID = 'spatialft/LFM2-350M-StepGame-ONNX'
api.create_repo(HF_REPO_ID, repo_type='model', exist_ok=True, private=False)

# Transformers.js expects:
#   root/  config.json, tokenizer.json, tokenizer_config.json, generation_config.json, ...
#   onnx/  model.onnx, model_fp16.onnx, model_q4.onnx

# Upload config/tokenizer files to root
for f in MERGED_DIR.iterdir():
    if f.suffix in ('.json', '.jinja', '.txt') and f.name != 'tokenizer.model':
        print(f'  uploading {f.name} → root')
        upload_file(path_or_fileobj=str(f), path_in_repo=f.name,
                    repo_id=HF_REPO_ID, repo_type='model')

# Upload ONNX files to onnx/ subfolder
for f in ONNX_DIR.glob('*.onnx'):
    print(f'  uploading {f.name} → onnx/{f.name}')
    upload_file(path_or_fileobj=str(f), path_in_repo=f'onnx/{f.name}',
                repo_id=HF_REPO_ID, repo_type='model')

print(f'\nDone. Model live at: https://huggingface.co/{HF_REPO_ID}')
print(f"To use in lfm2-web: MODEL_ID = '{HF_REPO_ID}', dtype = 'q4'")